# ARC hybrid: autoregressive proposer + TRM (z_H seeding)

Mirror of `hybrid_arc/` for Kaggle. Adjust `TRM_ROOT`, checkpoint, and data paths.

**Priors JSON**: base puzzle id → list of test output grids (see `hybrid_arc/priors.example.json` in the repo).


In [ ]:
import os, sys
from pathlib import Path
TRM_ROOT = Path("/kaggle/input/trm-code/TinyRecursiveModels-main")  # noqa: modify
NVARC = Path("/kaggle/working")
for p in (TRM_ROOT, NVARC):
    sys.path.insert(0, str(p))
os.environ.setdefault("DISABLE_COMPILE", "1")


In [ ]:
!uv pip install --no-deps --system hydra-core omegaconf pyyaml pydantic argdantic coolname tqdm adam-atan2-pytorch numba || true


In [ ]:
from pathlib import Path
(NVARC / "hybrid_arc").mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile hybrid_arc/__init__.py
"""
hybrid_arc: autoregressive teacher proposal + TRM inference with optional z_H warm-start.

Thesis-oriented method (see module docstrings in ``z_h_seed``, ``trm_z_h_seed``):

- **Core idea**: embed a prior output grid ``y_prior`` with the same ARC token layout as
  TRM labels, then add or blend those embeddings into ``z_H`` on grid positions (after
  ``puzzle_emb_len``) before the recurrent ``L_level`` blocks. This targets TRM's
  solution-track latent (paper ``y \\approx z_H``), not discrete teacher-forcing of
  logits between ACT steps.

**Limitations** (ablate and report):

1. **OOD seeding**: checkpoints were trained with ``H_init`` / ``L_init`` latents, not
   teacher-derived slices; use modest ``gamma`` and report sensitivity.
2. **``z_L`` not seeded**: wrong macro proposals can waste depth reconciling ``z_H``.
3. **Injection timing**: default seeds only when ``y_prior_tokens`` is present on the
   first ACT step (caller removes the key afterward if configured).
4. **Augmentation alignment**: ``y_prior`` must live in the same dihedral/color frame as
   the TRM dataloader row (use ``arc_tokenize.apply_row_augmentation`` from canonical).
"""

__all__ = ["arc_tokenize", "z_h_seed", "trm_z_h_seed", "llm_stage", "trm_stage", "pipeline", "run_local"]


In [ ]:
%%writefile hybrid_arc/arc_tokenize.py
"""
ARC grid ↔ TRM flat token layout (must match ``TRM/dataset/build_arc_dataset.py``).

TRM ``dataset`` imports are **lazy** so callers can prepend ``TRM/`` to ``sys.path`` first.
"""

from __future__ import annotations

from typing import Callable, Optional, Tuple

import numpy as np
import torch

# Duplicated constant to avoid importing ``dataset`` at module import time.
ARCMaxGridSize = 30


def flat_tokens_to_padded_grid(flat: np.ndarray) -> np.ndarray:
    """``flat`` length 900, values PAD=0, EOS=1, colors 2..11 as stored."""
    g = flat.reshape(ARCMaxGridSize, ARCMaxGridSize)
    return g


def crop_grid_from_padded(padded: np.ndarray) -> np.ndarray:
    """Maximum axis-aligned rectangle avoiding EOS (same spirit as evaluator ``_crop``)."""
    grid = padded.reshape(ARCMaxGridSize, ARCMaxGridSize)
    max_area = 0
    max_size = (0, 0)
    nr, nc = grid.shape
    num_c = nc
    for num_r in range(1, nr + 1):
        for c in range(1, num_c + 1):
            x = grid[num_r - 1, c - 1]
            if (x < 2) or (x > 11):
                num_c = c - 1
                break
        area = num_r * num_c
        if area > max_area:
            max_area = area
            max_size = (num_r, num_c)
    return (grid[: max_size[0], : max_size[1]] - 2).astype(np.uint8)


def input_flat_to_inp_grid(input_flat: np.ndarray) -> np.ndarray:
    """Decode TRM ``inputs`` row to uint8 input grid (0..9)."""
    padded = flat_tokens_to_padded_grid(np.asarray(input_flat, dtype=np.int64))
    return crop_grid_from_padded(padded)


def label_flat_from_grids(inp_grid: np.ndarray, out_grid: np.ndarray, do_translation: bool = False) -> np.ndarray:
    """Return label flat sequence (length 900) for ``out_grid`` paired with ``inp_grid``."""
    from dataset.build_arc_dataset import np_grid_to_seq_translational_augment

    _, out_flat = np_grid_to_seq_translational_augment(inp_grid, out_grid, do_translation=do_translation)
    return np.asarray(out_flat, dtype=np.int64)


def y_prior_to_label_flat(
    input_flat: np.ndarray,
    y_prior: np.ndarray,
    *,
    do_translation: bool = False,
) -> np.ndarray:
    """
    Build TRM label token ids for a proposal grid ``y_prior`` (HxW, 0..9) consistent
    with the padding/EOS layout implied by this row's ``input_flat``.
    """
    from dataset.build_arc_dataset import arc_grid_to_np

    inp = input_flat_to_inp_grid(input_flat)
    out = arc_grid_to_np(y_prior.tolist() if hasattr(y_prior, "tolist") else y_prior)
    return label_flat_from_grids(inp, out, do_translation=do_translation)


def row_augment_from_identifier_name(name: str) -> Tuple[str, Callable[[np.ndarray], np.ndarray]]:
    """``inverse_aug`` from TRM build_arc_dataset: maps augmented coords → canonical."""
    from dataset.build_arc_dataset import inverse_aug

    return inverse_aug(name)


def canonical_grid_to_row_space(grid: np.ndarray, augmented_identifier_name: str) -> np.ndarray:
    """
    Map a **canonical** (base puzzle) ``grid`` into the augmented frame used by TRM
    row ``augmented_identifier_name`` (must match ``identifiers.json`` entries).

    Mirrors ``dataset.build_arc_dataset.aug`` forward map:
    ``dihedral_transform(mapping[grid], trans_id)``.
    """
    from dataset.build_arc_dataset import PuzzleIdSeparator
    from dataset.common import dihedral_transform

    g = np.asarray(grid, dtype=np.uint8)
    if PuzzleIdSeparator not in augmented_identifier_name:
        return g
    parts = augmented_identifier_name.split(PuzzleIdSeparator)
    trans_id = int(parts[-2][1:])
    perm = parts[-1]
    mapping = np.array([int(c) for c in perm], dtype=np.uint8)
    if mapping.size != 10:
        raise ValueError(f"Expected 10 color-map digits in aug name, got {mapping.size}: {augmented_identifier_name!r}")
    return dihedral_transform(mapping[g], trans_id).astype(np.uint8)


def y_prior_tokens_torch(
    input_flat: np.ndarray,
    y_prior: np.ndarray,
    *,
    do_translation: bool = False,
    device: Optional[torch.device] = None,
) -> torch.Tensor:
    """``[1, seq_len]`` int64 token ids for embed_tokens."""
    flat = y_prior_to_label_flat(input_flat, y_prior, do_translation=do_translation)
    t = torch.from_numpy(flat).long().unsqueeze(0)
    if device is not None:
        t = t.to(device)
    return t


def batch_y_prior_tokens(
    inputs: torch.Tensor,
    y_priors: list[Optional[np.ndarray]],
    *,
    do_translation: bool = False,
    device: torch.device,
) -> Optional[torch.Tensor]:
    """
    Stack per-row ``y_prior`` into ``[B, seq_len]``. Rows with ``None`` use all -1
    (caller should disable seeding for that batch or filter).
    """
    if all(y is None for y in y_priors):
        return None
    rows = []
    for i, y in enumerate(y_priors):
        if y is None:
            rows.append(torch.full((inputs.shape[1],), -1, dtype=torch.long, device=device))
        else:
            flat = y_prior_to_label_flat(inputs[i].detach().cpu().numpy(), y, do_translation=do_translation)
            rows.append(torch.from_numpy(flat).long().to(device))
    return torch.stack(rows, dim=0)


In [ ]:
%%writefile hybrid_arc/z_h_seed.py
"""
Build embedding hints for ``z_H`` seeding from flat TRM label token ids.

Warm-start targets TRM's solution-track latent (``z_H`` after ``reset_carry``), in the
same subspace read by ``lm_head``. This is **not** the same as training-time teacher
forcing on discrete grids between ACT steps.
"""

from __future__ import annotations

from typing import Literal, Optional

import torch
import torch.nn as nn

SeedMode = Literal["add", "replace_blend"]


def embed_prior_grid_tokens(inner: nn.Module, y_prior_tokens: torch.Tensor) -> torch.Tensor:
    """
    ``y_prior_tokens``: ``[B, seq_len]`` int64 in TRM vocab (PAD=0, EOS=1, colors+2).
    Returns ``[B, seq_len, D]`` via ``inner.embed_tokens`` (same dtype as model).
    """
    return inner.embed_tokens(y_prior_tokens.to(torch.int32))


def apply_z_h_seed(
    z_H: torch.Tensor,
    hint: torch.Tensor,
    *,
    puzzle_emb_len: int,
    gamma: float,
    seed_mode: SeedMode,
    valid_mask: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    """
    Mutate ``z_H`` grid slice ``[:, puzzle_emb_len:, :]`` in-place.

    ``hint`` must match spatial size ``z_H[:, puzzle_emb_len:, :].shape``.

    ``valid_mask`` optional ``[B, seq_len]`` bool; False positions skip update (keeps
    initialized ``z_H`` there). If ``None``, all positions updated.
    """
    sl = slice(puzzle_emb_len, None)
    zg = z_H[:, sl, :]
    hg = hint
    if hg.shape != zg.shape:
        raise ValueError(f"hint shape {hg.shape} != z_H grid slice {zg.shape}")
    if valid_mask is None:
        m = torch.ones(zg.shape[:2], dtype=torch.bool, device=zg.device)
    else:
        m = valid_mask
    m3 = m.unsqueeze(-1)
    g = float(gamma)
    if seed_mode == "add":
        zg[:] = torch.where(m3, zg + g * hg, zg)
    elif seed_mode == "replace_blend":
        zg[:] = torch.where(m3, (1.0 - g) * zg + g * hg, zg)
    else:
        raise ValueError(seed_mode)
    return z_H


In [ ]:
%%writefile hybrid_arc/trm_z_h_seed.py
"""
Subclass TRM inner module to inject teacher proposal embeddings into ``z_H``.

See package docstring in ``hybrid_arc.__init__`` for thesis limitations.
"""

from __future__ import annotations

from typing import Dict, Tuple

import torch
from torch import nn

from models.recursive_reasoning.trm import (
    TinyRecursiveReasoningModel_ACTV1_Inner,
    TinyRecursiveReasoningModel_ACTV1InnerCarry,
)

from .z_h_seed import SeedMode, apply_z_h_seed, embed_prior_grid_tokens


class TinyRecursiveReasoningModel_ACTV1_Inner_ZHSeed(TinyRecursiveReasoningModel_ACTV1_Inner):
    """
    Same weights as ``TinyRecursiveReasoningModel_ACTV1_Inner``, with optional
    ``batch["y_prior_tokens"]`` of shape ``[B, seq_len]`` (``-1`` = skip position for
    per-row masking when stacking mixed batches).
    """

    def __init__(self, config, *, gamma: float = 0.1, seed_mode: SeedMode = "add") -> None:
        super().__init__(config)
        self.gamma = float(gamma)
        self.seed_mode: SeedMode = seed_mode

    def forward(
        self,
        carry: TinyRecursiveReasoningModel_ACTV1InnerCarry,
        batch: Dict[str, torch.Tensor],
    ) -> Tuple[TinyRecursiveReasoningModel_ACTV1InnerCarry, torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        seq_info = dict(
            cos_sin=self.rotary_emb() if hasattr(self, "rotary_emb") else None,
        )
        input_embeddings = self._input_embeddings(batch["inputs"], batch["puzzle_identifiers"])

        z_H, z_L = carry.z_H, carry.z_L

        y_prior_tokens = batch.get("y_prior_tokens")
        if y_prior_tokens is not None and self.gamma > 0.0:
            valid = y_prior_tokens >= 0
            hint = embed_prior_grid_tokens(self, y_prior_tokens.clamp(min=0))
            z_H = apply_z_h_seed(
                z_H,
                hint,
                puzzle_emb_len=self.puzzle_emb_len,
                gamma=self.gamma,
                seed_mode=self.seed_mode,
                valid_mask=valid,
            )

        with torch.no_grad():
            for _H_step in range(self.config.H_cycles - 1):
                for _L_step in range(self.config.L_cycles):
                    z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
                z_H = self.L_level(z_H, z_L, **seq_info)
        for _L_step in range(self.config.L_cycles):
            z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
        z_H = self.L_level(z_H, z_L, **seq_info)

        new_carry = TinyRecursiveReasoningModel_ACTV1InnerCarry(z_H=z_H.detach(), z_L=z_L.detach())
        output = self.lm_head(z_H)[:, self.puzzle_emb_len :]
        q_logits = self.q_head(z_H[:, 0]).to(torch.float32)
        return new_carry, output, (q_logits[..., 0], q_logits[..., 1])


def _module_device(module: nn.Module) -> torch.device:
    """Device of first parameter or buffer (inner may be buffer-heavy)."""
    p = next(module.parameters(), None)
    if p is not None:
        return p.device
    b = next(module.buffers(), None)
    if b is not None:
        return b.device
    return torch.device("cpu")


def patch_act_model_inner(model: nn.Module, *, gamma: float, seed_mode: SeedMode) -> TinyRecursiveReasoningModel_ACTV1_Inner_ZHSeed:
    """
    Replace ``model.model.inner`` on an ``ACTLossHead``-wrapped TRM.

    ``model`` is typically ``train_state.model`` from ``eval-arc-k-10`` (``ACTLossHead``).
    """
    act = model
    trm = act.model  # TinyRecursiveReasoningModel_ACTV1
    cfg = trm.config
    device = _module_device(trm.inner)
    new_inner = TinyRecursiveReasoningModel_ACTV1_Inner_ZHSeed(cfg, gamma=gamma, seed_mode=seed_mode)
    new_inner.load_state_dict(trm.inner.state_dict(), strict=True)
    new_inner.to(device)
    trm.inner = new_inner
    return new_inner


In [ ]:
%%writefile hybrid_arc/llm_stage.py
"""
Pluggable autoregressive teacher: produce ``y_prior`` (HxW uint8) or ``None``.

Implementations may wrap HuggingFace, Unsloth, vLLM, etc.; the **interface** stays
model-agnostic so the thesis and ``pipeline`` do not depend on a vendor name.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Protocol


@dataclass
class ArcTestQuery:
    """One test cell: train pairs + single test input (ARC JSON shape)."""

    puzzle_id: str
    train_pairs: List[Dict[str, Any]]
    test_input: List[List[int]]


class AutoregressiveTeacher(Protocol):
    """Callable contract for ``hybrid_arc.pipeline``."""

    def propose(self, query: ArcTestQuery) -> Optional[Any]:
        """Return ``numpy.ndarray`` uint8 HxW (0..9), or ``None`` if unavailable."""


class NullTeacher:
    """Always returns ``None`` (TRM-only / ablation)."""

    def propose(self, query: ArcTestQuery) -> Optional[Any]:
        return None


def numpy_if_valid(arr: Any) -> Optional[Any]:
    """Light validation for uint8 grids."""
    import numpy as np

    if arr is None:
        return None
    a = np.asarray(arr, dtype=np.uint8)
    if a.ndim != 2 or a.shape[0] < 1 or a.shape[0] > 30 or a.shape[1] < 1 or a.shape[1] > 30:
        return None
    if not np.all((a >= 0) & (a <= 9)):
        return None
    return a


In [ ]:
%%writefile hybrid_arc/trm_stage.py
"""
Load TRM + ACT head, optionally patch inner for ``z_H`` seeding, run test inference.

Requires ``TRM`` on ``sys.path`` (see ``run_local`` / ``python -m hybrid_arc.run_local``).
"""

from __future__ import annotations

import importlib.util
import json
import os
import sys
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

import torch
from omegaconf import DictConfig, OmegaConf

from .arc_tokenize import batch_y_prior_tokens
from .trm_z_h_seed import SeedMode, patch_act_model_inner


def _load_eval_module(trm_root: Path):
    path = trm_root / "eval-arc-k-10.py"
    spec = importlib.util.spec_from_file_location("eval_arc_k10", path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot load {path}")
    mod = importlib.util.module_from_spec(spec)
    sys.modules["eval_arc_k10"] = mod
    spec.loader.exec_module(mod)
    return mod


def load_identifiers(data_dir: Path) -> List[str]:
    path = data_dir / "identifiers.json"
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def compose_pretrain_config(trm_root: Path, config_name: str, overrides: List[str]) -> Any:
    from hydra import compose, initialize_config_dir

    with initialize_config_dir(version_base=None, config_dir=str(trm_root / "config")):
        cfg: DictConfig = compose(config_name=config_name, overrides=overrides)
    d = OmegaConf.to_container(cfg, resolve=True)
    eak = _load_eval_module(trm_root)
    pc = eak.PretrainConfig
    mv = getattr(pc, "model_validate", None)
    if callable(mv):
        return mv(d)  # type: ignore[no-any-return]
    return pc(**d)  # type: ignore[arg-type]


def evaluate_with_z_h_seed(
    *,
    trm_root: Path,
    config: Any,
    train_state: Any,
    eval_loader: Any,
    eval_metadata: Any,
    evaluators: List[Any],
    rank: int,
    world_size: int,
    cpu_group: Optional[Any],
    identifier_list: List[str],
    prior_batch_fn: Optional[Callable[[Dict[str, torch.Tensor], List[str]], List[Optional[Any]]]],
    seed_first_step_only: bool,
) -> Optional[Dict[str, Any]]:
    """
    Same as ``evaluate`` in ``eval-arc-k-10.py``, but optional ``y_prior_tokens`` on the
    first ACT iteration per batch when ``prior_batch_fn`` returns grids.
    """
    reduced_metrics = None

    with torch.inference_mode():
        return_keys = set(config.eval_save_outputs)
        for evaluator in evaluators:
            evaluator.begin_eval()
            return_keys.update(evaluator.required_outputs)

        set_ids = {k: idx for idx, k in enumerate(eval_metadata.sets)}
        save_preds: Dict[str, List[torch.Tensor]] = {}
        metric_keys: List[str] = []
        metric_values: Optional[torch.Tensor] = None
        processed_batches = 0

        for set_name, batch, global_batch_size in eval_loader:
            processed_batches += 1
            if rank == 0:
                print(f"Processing batch {processed_batches}: {set_name}")

            batch = {k: v.cuda() for k, v in batch.items()}
            id_tensor = batch["puzzle_identifiers"]
            row_names = [identifier_list[int(i)] for i in id_tensor.cpu().tolist()]

            if prior_batch_fn is not None:
                y_list = prior_batch_fn(batch, row_names)
                yt = batch_y_prior_tokens(
                    batch["inputs"],
                    y_list,
                    do_translation=False,
                    device=batch["inputs"].device,
                )
                if yt is not None:
                    batch["y_prior_tokens"] = yt

            with torch.device("cuda"):
                carry = train_state.model.initial_carry(batch)  # type: ignore[operator]

            inference_steps = 0
            while True:
                if seed_first_step_only and inference_steps > 0:
                    batch.pop("y_prior_tokens", None)
                carry, loss, metrics, preds, all_finish = train_state.model(
                    carry=carry, batch=batch, return_keys=return_keys
                )
                inference_steps += 1
                if all_finish:
                    break

            if rank == 0:
                print(f"  Completed inference in {inference_steps} steps")

            for collection in (batch, preds):
                for k, v in collection.items():
                    if k in config.eval_save_outputs:
                        save_preds.setdefault(k, [])
                        save_preds[k].append(v.cpu())

            for evaluator in evaluators:
                evaluator.update_batch(batch, preds)

            del carry, loss, preds, batch, all_finish

            set_id = set_ids[set_name]
            if metric_values is None:
                metric_keys = list(sorted(metrics.keys()))
                metric_values = torch.zeros(
                    (len(set_ids), len(metrics.values())), dtype=torch.float32, device="cuda"
                )
            metric_values[set_id] += torch.stack([metrics[k] for k in metric_keys])
            del metrics

        save_preds = {k: torch.cat(v, dim=0) for k, v in save_preds.items()}

        if config.checkpoint_path is not None and len(save_preds):
            os.makedirs(os.path.dirname(config.checkpoint_path), exist_ok=True)
            torch.save(
                save_preds,
                os.path.join(config.checkpoint_path, f"step_{train_state.step}_all_preds.{rank}"),
            )
        del save_preds

        if metric_values is not None:
            if world_size > 1:
                import torch.distributed as dist  # noqa: WPS433

                if dist.is_initialized():
                    dist.reduce(metric_values, dst=0)
            if rank == 0:
                reduced_metrics = metric_values.cpu().numpy()
                reduced_metrics = {
                    set_name: {
                        metric_name: reduced_metrics[set_id, metric_id]
                        for metric_id, metric_name in enumerate(metric_keys)
                    }
                    for set_id, set_name in enumerate(set_ids)
                }
                for set_name, m in reduced_metrics.items():
                    count = m.pop("count")
                    reduced_metrics[set_name] = {k: v / count for k, v in m.items()}

        if rank == 0:
            print(f"\nRunning {len(evaluators)} evaluator(s)...")
        for i, evaluator in enumerate(evaluators):
            if rank == 0:
                print(f"Running evaluator {i+1}/{len(evaluators)}: {evaluator.__class__.__name__}")
            evaluator_save_path = None
            if config.checkpoint_path is not None:
                evaluator_save_path = os.path.join(
                    config.checkpoint_path,
                    f"evaluator_{evaluator.__class__.__name__}_step_{train_state.step}",
                )
                os.makedirs(evaluator_save_path, exist_ok=True)
            metrics = evaluator.result(evaluator_save_path, rank=rank, world_size=world_size, group=cpu_group)
            if rank == 0 and metrics is not None:
                if reduced_metrics is None:
                    reduced_metrics = {}
                reduced_metrics.update(metrics)
                print(f"  Completed {evaluator.__class__.__name__}")
        if rank == 0:
            print("All evaluators completed!")

    return reduced_metrics


def run_hybrid_eval(
    *,
    trm_root: Path,
    config_overrides: List[str],
    load_checkpoint: str,
    gamma: float,
    seed_mode: SeedMode,
    use_z_h_seed: bool,
    prior_batch_fn: Optional[Callable[[Dict[str, torch.Tensor], List[str]], List[Optional[Any]]]],
    seed_first_step_only: bool = True,
    data_dir_for_identifiers: Optional[Path] = None,
) -> Optional[Dict[str, Any]]:
    """
    End-to-end hybrid eval: compose Hydra config, build loaders, load weights, patch inner.
    """
    os.environ.setdefault("DISABLE_COMPILE", "1")
    trm_root = trm_root.resolve()
    if str(trm_root) not in sys.path:
        sys.path.insert(0, str(trm_root))

    eak = _load_eval_module(trm_root)
    config = compose_pretrain_config(trm_root, "cfg_pretrain", config_overrides)
    config.load_checkpoint = load_checkpoint

    RANK = 0
    WORLD_SIZE = 1
    CPU_PROCESS_GROUP = None

    _train_loader, train_metadata = eak.create_dataloader(
        config,
        "train",
        test_set_mode=False,
        epochs_per_iter=config.eval_interval if config.eval_interval is not None else config.epochs,
        global_batch_size=config.global_batch_size,
        rank=RANK,
        world_size=WORLD_SIZE,
    )
    eval_loader, eval_metadata = eak.create_dataloader(
        config,
        "test",
        test_set_mode=True,
        epochs_per_iter=1,
        global_batch_size=config.global_batch_size,
        rank=RANK,
        world_size=WORLD_SIZE,
    )

    evaluators = eak.create_evaluators(config, eval_metadata)

    train_state = eak.init_train_state(config, train_metadata, rank=RANK, world_size=WORLD_SIZE)
    train_state.model.eval()

    if use_z_h_seed and gamma > 0:
        patch_act_model_inner(train_state.model, gamma=gamma, seed_mode=seed_mode)

    ident_path = data_dir_for_identifiers
    if ident_path is None and config.data_paths_test:
        ident_path = Path(config.data_paths_test[0])
    elif ident_path is None:
        ident_path = Path(config.data_paths[0])
    identifier_list = load_identifiers(ident_path)

    return evaluate_with_z_h_seed(
        trm_root=trm_root,
        config=config,
        train_state=train_state,
        eval_loader=eval_loader,
        eval_metadata=eval_metadata,
        evaluators=evaluators,
        rank=RANK,
        world_size=WORLD_SIZE,
        cpu_group=CPU_PROCESS_GROUP,
        identifier_list=identifier_list,
        prior_batch_fn=prior_batch_fn if use_z_h_seed else None,
        seed_first_step_only=seed_first_step_only,
    )


In [ ]:
%%writefile hybrid_arc/pipeline.py
"""
Orchestration: optional teacher priors, TRM eval with ``z_H`` seeding, CSV ablation logs.
"""

from __future__ import annotations

import csv
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional

import numpy as np

from .arc_tokenize import canonical_grid_to_row_space
from .trm_stage import SeedMode, run_hybrid_eval


@dataclass
class PriorStore:
    """Load ``priors.json``: ``{ base_puzzle_id: [ per-test-index list of grids ] }``."""

    data: Dict[str, List[Any]]

    @classmethod
    def from_path(cls, path: Path) -> "PriorStore":
        with open(path, "r", encoding="utf-8") as f:
            return cls(json.load(f))

    def get(self, augmented_name: str, test_flat_index: int = 0) -> Optional[np.ndarray]:
        from dataset.build_arc_dataset import PuzzleIdSeparator  # noqa: WPS433

        sep = PuzzleIdSeparator
        base = augmented_name.split(sep)[0] if sep in augmented_name else augmented_name
        if base not in self.data:
            return None
        grids = self.data[base]
        if test_flat_index < 0 or test_flat_index >= len(grids):
            return None
        g = grids[test_flat_index]
        if g is None:
            return None
        a = np.asarray(g, dtype=np.uint8)
        if a.ndim != 2:
            return None
        if sep in augmented_name:
            a = canonical_grid_to_row_space(a, augmented_name)
        return a


def make_prior_fn(store: Optional[PriorStore]) -> Optional[Callable[..., List[Optional[Any]]]]:
    if store is None:
        return None

    def _fn(batch, row_names: List[str]) -> List[Optional[Any]]:
        out: List[Optional[Any]] = []
        for name in row_names:
            out.append(store.get(name, 0))
        return out

    return _fn


def log_ablation_row(csv_path: Path, row: Dict[str, Any]) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not csv_path.exists()
    with open(csv_path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_header:
            w.writeheader()
        w.writerow(row)


def run_mode(
    *,
    trm_root: Path,
    overrides: List[str],
    checkpoint: str,
    mode: str,
    gamma: float,
    seed_mode: SeedMode,
    priors_path: Optional[Path],
    csv_path: Optional[Path],
    data_dir_identifiers: Optional[Path],
) -> Optional[Dict[str, Any]]:
    """
    ``mode``: ``trm_only`` | ``z_h_seed`` | ``z_h_seed_zero`` (gamma 0, patched inner).
    """
    store = PriorStore.from_path(priors_path) if priors_path and priors_path.exists() else None
    prior_fn = make_prior_fn(store)

    use_seed = mode in ("z_h_seed", "z_h_seed_zero")
    g = 0.0 if mode == "z_h_seed_zero" else float(gamma)
    metrics = run_hybrid_eval(
        trm_root=trm_root,
        config_overrides=overrides,
        load_checkpoint=checkpoint,
        gamma=g,
        seed_mode=seed_mode,
        use_z_h_seed=use_seed,
        prior_batch_fn=prior_fn,
        seed_first_step_only=True,
        data_dir_for_identifiers=data_dir_identifiers,
    )
    if csv_path is not None:
        flat: Dict[str, Any] = {"mode": mode, "gamma": g, "seed_mode": seed_mode}
        if metrics:
            for split, m in metrics.items():
                for k, v in m.items():
                    flat[f"{split}/{k}"] = float(v) if hasattr(v, "item") else v
        log_ablation_row(csv_path, flat)
    return metrics


In [ ]:
%%writefile hybrid_arc/run_local.py
"""
Local entrypoint for hybrid ARC eval (``z_H`` seeding + optional priors JSON).

Run from repository root::

    python -m hybrid_arc.run_local --trm-root TRM --checkpoint path/to/step_N \\
        --overrides data_paths_test=[path/to/arc2test-aug-128] '+load_checkpoint=...' \\
        --modes trm_only,z_h_seed --gamma 0.1 --seed-mode add \\
        --priors-json optional.json --ablation-csv results.csv

Requires CUDA (same as ``TRM/eval-arc-k-10.py``). Set ``DISABLE_COMPILE=1`` (default in code).
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path


def _prepend_sys_path() -> Path:
    root = Path(__file__).resolve().parents[1]
    trm = root / "TRM"
    for p in (trm, root):
        s = str(p)
        if s not in sys.path:
            sys.path.insert(0, s)
    return root


def main() -> None:
    ap = argparse.ArgumentParser(description="Hybrid ARC: TRM + optional z_H seed from priors JSON")
    ap.add_argument("--trm-root", type=Path, default=Path("TRM"), help="Path to TRM package (contains eval-arc-k-10.py, config/)")
    ap.add_argument("--checkpoint", type=str, required=True, help="Checkpoint file passed as load_checkpoint")
    ap.add_argument(
        "--overrides",
        type=str,
        nargs="*",
        default=[],
        help='Hydra overrides, e.g. data_paths_test=["D:/data/arc2test-aug-128"]',
    )
    ap.add_argument(
        "--modes",
        type=str,
        default="trm_only",
        help="Comma-separated: trm_only | z_h_seed | z_h_seed_zero",
    )
    ap.add_argument("--gamma", type=float, default=0.1)
    ap.add_argument("--seed-mode", type=str, default="add", choices=("add", "replace_blend"))
    ap.add_argument("--priors-json", type=Path, default=None, help="Optional priors JSON for z_h_seed (see hybrid_arc.pipeline.PriorStore)")
    ap.add_argument("--ablation-csv", type=Path, default=None, help="Append one row per mode with aggregate metrics")
    ap.add_argument("--identifiers-dir", type=Path, default=None, help="Directory containing identifiers.json (default: first data_paths_test or data_paths)")

    args = ap.parse_args()
    _prepend_sys_path()

    def _has_load_checkpoint_override(ovr: list[str]) -> bool:
        for o in ovr:
            rest = o[1:] if o.startswith("+") else o
            if rest.startswith("load_checkpoint="):
                return True
        return False

    overrides = list(args.overrides)
    if args.checkpoint and not _has_load_checkpoint_override(overrides):
        overrides.append(f"+load_checkpoint={args.checkpoint}")

    from hybrid_arc.pipeline import run_mode
    from hybrid_arc.trm_stage import SeedMode

    sm: SeedMode = "replace_blend" if args.seed_mode == "replace_blend" else "add"
    modes = [m.strip() for m in args.modes.split(",") if m.strip()]

    for mode in modes:
        print(f"=== mode={mode} ===")
        run_mode(
            trm_root=args.trm_root.resolve(),
            overrides=overrides,
            checkpoint=args.checkpoint,
            mode=mode,
            gamma=args.gamma,
            seed_mode=sm,
            priors_path=args.priors_json,
            csv_path=args.ablation_csv,
            data_dir_identifiers=args.identifiers_dir,
        )


if __name__ == "__main__":
    main()


In [ ]:
from pathlib import Path
from hybrid_arc.pipeline import run_mode

CKPT = Path("/kaggle/input/your-trm-checkpoint/step_100000")  # noqa: modify
DATA = Path("/kaggle/input/your-arc2test-aug-128")
overrides = [f'data_paths_test=[\"{DATA.as_posix()}\"]', 'global_batch_size=8']
run_mode(
    trm_root=TRM_ROOT,
    overrides=overrides,
    checkpoint=str(CKPT),
    mode="trm_only",
    gamma=0.1,
    seed_mode="add",
    priors_path=None,
    csv_path=Path("/kaggle/working/ablation.csv"),
    data_dir_identifiers=DATA,
)
